# Pretraining notebook

## Section 1: Generate text before training

Load your current model

Enter a few prompts and note the output.

Try these:
 - love is
 - baby
 - tonight

Generate 20 tokens and note the output.

Reflection: Is there any quality or meaning to the output or is it largely random text?  Why?

In [1]:
import sys
from pathlib import Path

import torch
import tiktoken


PROJECT_ROOT = Path.cwd().parent
LAB5_DIR = PROJECT_ROOT / "lab5"
MODEL_PATH = PROJECT_ROOT / "model.pth"

if not LAB5_DIR.is_dir():
    raise FileNotFoundError(
        f"lab5 folder not found:\n{LAB5_DIR.resolve()}"
    )

if not (LAB5_DIR / "previous_chapters.py").is_file():
    raise FileNotFoundError(
        "previous_chapters.py not found:\n"
        f"{(LAB5_DIR / 'previous_chapters.py').resolve()}"
    )

if not MODEL_PATH.is_file():
    raise FileNotFoundError(
        f"model.pth not found:\n{MODEL_PATH.resolve()}"
    )

if str(LAB5_DIR) not in sys.path:
    sys.path.insert(0, str(LAB5_DIR))

from previous_chapters import GPTModel


torch.manual_seed(123)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Using device: {device}")


MODEL_CONFIG = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}


model = GPTModel(MODEL_CONFIG)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
    weights_only=True,
)

model.load_state_dict(checkpoint)
model.to(device)
model.eval()

tokenizer = tiktoken.get_encoding("gpt2")


def text_to_token_ids(text, tokenizer):
    token_ids = tokenizer.encode(
        text,
        allowed_special={"<|endoftext|>"},
    )
    return torch.tensor(
        token_ids,
        dtype=torch.long,
    ).unsqueeze(0)


def token_ids_to_text(token_ids, tokenizer):
    return tokenizer.decode(token_ids.squeeze(0).tolist())


@torch.no_grad()
def generate(
    model,
    idx,
    max_new_tokens,
    context_size,
    temperature=0.8,
    top_k=50,
):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        logits = model(idx_cond)
        logits = logits[:, -1, :]

        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            cutoff = top_logits[:, -1].unsqueeze(-1)

            logits = torch.where(
                logits < cutoff,
                torch.tensor(
                    float("-inf"),
                    device=logits.device,
                    dtype=logits.dtype,
                ),
                logits,
            )

        if temperature == 0.0:
            idx_next = torch.argmax(
                logits,
                dim=-1,
                keepdim=True,
            )
        else:
            probabilities = torch.softmax(
                logits / temperature,
                dim=-1,
            )
            idx_next = torch.multinomial(
                probabilities,
                num_samples=1,
            )

        idx = torch.cat((idx, idx_next), dim=1)

    return idx


prompts = [
    "love is",
    "baby",
    "tonight",
]


for prompt in prompts:
    torch.manual_seed(123)

    input_token_ids = text_to_token_ids(
        prompt,
        tokenizer,
    ).to(device)

    output_token_ids = generate(
        model=model,
        idx=input_token_ids,
        max_new_tokens=40,
        context_size=MODEL_CONFIG["context_length"],
        temperature=0.8,
        top_k=50,
    )

    output_text = token_ids_to_text(
        output_token_ids,
        tokenizer,
    )

    print("\n" + "=" * 72)
    print(f"Prompt: {prompt!r}")
    print("-" * 72)
    print(output_text)

Using device: cpu

Prompt: 'love is'
------------------------------------------------------------------------
love is not Of course Int cheeks furrowed by a smile that lifted the tips of a self-confident moustache, I felt to what a degree he had the glory, and his pictures

Prompt: 'baby'
------------------------------------------------------------------------
babyas he had been through, and in spite of, his pictures--so handsome, so charming, so disarming, that one longed to cry out: "Be dissatisfied with your leisure!" as

Prompt: 'tonight'
------------------------------------------------------------------------
tonight much amazement escapent cheeks furrowed by a smile that lifted the tips of a self-confident moustache, I felt to what a degree he had the glory, and his pictures


## Section 2: Understanding Logits

Using the logits starter code, do the following:
 - Apply softmax
 - Identify the highest probability token

Reflection: Why is softmax necessary before interpreting logits as probabilities?

In [2]:
logits = torch.tensor([
    [1.5, 2.0, 0.5]
])

probabilities = torch.softmax(logits, dim=-1)

print(probabilities)
print(probabilities.sum())

predicted_token = torch.argmax(probabilities)

print(predicted_token)

tensor([[0.3315, 0.5465, 0.1220]])
tensor(1.0000)
tensor(1)


## Section 3: Cross Entropy Loss

### Textbook 
The textbook discussion begins on page 136

### Manually calculate
 - Probabilities
 - Log probabilities
 - Average negative log probabilities

### Compare with torch
Compare your manual results to the torch cross entropy results

```python
     torch.nn.functional.cross_entropy(...)
'''


In [3]:
import torch

# Manually calculate values

probs = torch.tensor([
    0.7,
    0.2,
    0.1
])

target_index = 0

In [4]:
target_probability = probs[target_index]

print(target_probability)

tensor(0.7000)


In [5]:
log_probability = torch.log(target_probability)

print(log_probability)

tensor(-0.3567)


In [6]:
loss = -log_probability

print(loss)

tensor(0.3567)


In [7]:
import torch
import torch.nn.functional as F

torch.set_printoptions(precision=8, sci_mode=False)

logits = torch.tensor(
    [
        [2.0, 1.0, 0.1],
        [0.5, 2.5, 1.0],
        [1.2, 0.3, 2.1],
        [-1.0, 0.0, 1.0],
    ],
    dtype=torch.float64,
    requires_grad=True,
)

targets = torch.tensor([0, 1, 2, 1], dtype=torch.long)

log_probs = F.log_softmax(logits, dim=1)
per_example_manual = -log_probs[torch.arange(targets.shape[0]), targets]
loss_manual_none = per_example_manual
loss_manual_sum = per_example_manual.sum()
loss_manual_mean = per_example_manual.mean()

loss_torch_none = F.cross_entropy(logits, targets, reduction="none")
loss_torch_sum = F.cross_entropy(logits, targets, reduction="sum")
loss_torch_mean = F.cross_entropy(logits, targets, reduction="mean")

print("Logits:")
print(logits.detach())
print()
print("Targets:")
print(targets)
print()
print("Softmax probabilities:")
print(torch.softmax(logits.detach(), dim=1))
print()
print("Log probabilities:")
print(log_probs.detach())
print()
print("Manual loss, reduction='none':")
print(loss_manual_none.detach())
print()
print("torch loss, reduction='none':")
print(loss_torch_none.detach())
print()
print("Manual loss, reduction='sum':")
print(loss_manual_sum.detach())
print()
print("torch loss, reduction='sum':")
print(loss_torch_sum.detach())
print()
print("Manual loss, reduction='mean':")
print(loss_manual_mean.detach())
print()
print("torch loss, reduction='mean':")
print(loss_torch_mean.detach())
print()
print("Elementwise match:")
print(torch.allclose(loss_manual_none, loss_torch_none))
print()
print("Sum match:")
print(torch.allclose(loss_manual_sum, loss_torch_sum))
print()
print("Mean match:")
print(torch.allclose(loss_manual_mean, loss_torch_mean))

loss_torch_mean.backward()

print()
print("Gradient of logits from torch cross entropy:")
print(logits.grad)

Logits:
tensor([[ 2.00000000,  1.00000000,  0.10000000],
        [ 0.50000000,  2.50000000,  1.00000000],
        [ 1.20000000,  0.30000000,  2.10000000],
        [-1.00000000,  0.00000000,  1.00000000]], dtype=torch.float64)

Targets:
tensor([0, 1, 2, 1])

Softmax probabilities:
tensor([[0.65900114, 0.24243297, 0.09856589],
        [0.09962365, 0.73612472, 0.16425163],
        [0.25865373, 0.10516076, 0.63618551],
        [0.09003057, 0.24472847, 0.66524096]], dtype=torch.float64)

Log probabilities:
tensor([[-0.41703002, -1.41703002, -2.31703002],
        [-2.30635571, -0.30635571, -1.80635571],
        [-1.35226507, -2.25226507, -0.45226507],
        [-2.40760596, -1.40760596, -0.40760596]], dtype=torch.float64)

Manual loss, reduction='none':
tensor([0.41703002, 0.30635571, 0.45226507, 1.40760596], dtype=torch.float64)

torch loss, reduction='none':
tensor([0.41703002, 0.30635571, 0.45226507, 1.40760596], dtype=torch.float64)

Manual loss, reduction='sum':
tensor(2.58325676, dtype=

## Section 4: Perplexity

Compute the perplexity

```python
perplexity = torch.exp(loss)
```

**Note:** A perplexity of 10 means the model is roughly as uncertain as choosing among 10 equally likely next tokens.

In [8]:
loss = torch.tensor(2.5)

perplexity = torch.exp(loss)

print(perplexity)

tensor(12.18249416)


## Section 5: Understanding the Training Loop

The training loop performs:

1. Iterate through epochs
2. Iterate through batches
3. Zero gradients
4. Calculate loss
5. Backpropagation
6. Optimizer step
7. Evaluate model
8. Generate sample text

Draw or explain this process in your own words.

# Training vs. Validation

You will:
 - Split lyrics into train/validation
 - Create loaders
 - Compute initial loss

Keep note of your: 
 - Training Loss: 
 - Validation Loss: 

In [24]:
from weird_ai.dataset import LyricsDataset

with open("lyrics.txt", "r", encoding="utf-8") as f:
    corpus = f.read()

train_ratio = 0.9

split_idx = int(len(corpus) * train_ratio)
split_idx = corpus.rfind("\n", 0, split_idx)  # optional: split at a line break

train_text = corpus[:split_idx]
val_text = corpus[split_idx:]

print("Corpus characters:", len(corpus))
print("Training characters:", len(train_text))
print("Validation characters:", len(val_text))

FileNotFoundError: [Errno 2] No such file or directory: 'lyrics.txt'

In [19]:
from torch.utils.data import DataLoader

context_length = 128
batch_size = 64

train_dataset = LyricsDataset(train_text, context_length=context_length)
val_dataset = LyricsDataset(val_text, context_length=context_length)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
)

print(train_loader)
print(val_loader)

NameError: name 'train_text' is not defined

In [ ]:
import torch
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

@torch.no_grad()
def calculate_loss(loader, model):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            y.reshape(-1),
            reduction="sum",
        )

        total_loss += loss.item()
        total_tokens += y.numel()

    model.train()
    return total_loss / total_tokens

train_loss = calculate_loss(train_loader, model)
val_loss = calculate_loss(val_loader, model)

print(f"Initial training loss:   {train_loss:.4f}")
print(f"Initial validation loss: {val_loss:.4f}")

## Section 6: Training

Train for one Epoch
```python
train_model(...)
```

Then record the loss before and after training.  

In [ ]:
num_epochs = 1

# TODO:
# Train the model

# TODO:
# Record:
# Training loss before
# Training loss after

In [ ]:
print(f"Initial loss: {initial_loss}")
print(f"Final loss: {final_loss}")

## Evaluation

Did the loss decrease?

Why is decreasing loss important?

## Before and After Comparison

Prompt:

love is

Before Training:
__________________

After Training:
__________________

Reflection:

How did the output change?

What evidence do you see that the model learned something from the training data?